# Task 1 — SFT

## Setup

In [ ]:
%pip install -q \
  "transformers==5.13.1" \
  "trl==1.8.0" \
  "peft==0.19.1" \
  "bitsandbytes==0.49.2" \
  "datasets==5.0.0" \
  "accelerate==1.14.0" \
  "scikit-learn==1.7.2" \
  "scipy>=1.12,<2" \
  "pandas>=2.2,<3" \
  "safetensors>=0.5"

In [ ]:
import gc
import json
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.sparse import hstack
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

assert torch.cuda.is_available()

print(torch.cuda.get_device_name(0))
print(torch.__version__)

## Data

In [ ]:
def find_file(filename: str, folder: str | None = None) -> Path:
    roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/task1_sft_colab"),
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    candidates = []

    for root in roots:
        if folder is not None:
            candidates.append(root / folder / filename)
        candidates.append(root / filename)

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    for root in roots:
        if root.exists():
            matches = list(root.glob(f"**/{filename}"))
            if matches:
                return matches[0].resolve()

    raise FileNotFoundError(filename)


def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


TRAIN_PATH = find_file("kid_adult.jsonl", "data")
STYLE_TEST_PATH = find_file("public_test_style.jsonl", "data")
STYLE_CLF_PATH = find_file("style_clf.pkl", "metrics")

OUTPUT_DIR = Path("/content/outputs/task1_sft")
if not Path("/content").exists():
    OUTPUT_DIR = Path.cwd() / "outputs" / "task1_sft"

ADAPTER_DIR = OUTPUT_DIR / "adapter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_rows = read_jsonl(TRAIN_PATH)
style_test_rows = read_jsonl(STYLE_TEST_PATH)

assert len(train_rows) == 1489
assert len(style_test_rows) == 50
assert {"prompt", "kid", "adult"} <= train_rows[0].keys()
assert {"prompt", "kid", "adult", "fact"} <= style_test_rows[0].keys()

print(TRAIN_PATH)
print(STYLE_TEST_PATH)
print(STYLE_CLF_PATH)
print(len(train_rows), len(style_test_rows))

In [ ]:
with STYLE_CLF_PATH.open("rb") as file:
    style_bundle = pickle.load(file)

style_clf = style_bundle["clf"]
style_vecs = style_bundle["vecs"]


def style_probabilities(texts: list[str]) -> np.ndarray:
    features = hstack(
        [vectorizer.transform(texts) for vectorizer in style_vecs],
        format="csr",
    )
    probabilities = style_clf.predict_proba(features)
    simple_column = int(np.where(style_clf.classes_ == 1)[0][0])
    return probabilities[:, simple_column]


gold_kid_p = float(
    style_probabilities([row["kid"] for row in style_test_rows]).mean()
)
gold_adult_p = float(
    style_probabilities([row["adult"] for row in style_test_rows]).mean()
)

assert gold_kid_p > 0.9
assert gold_adult_p < 0.1

print(f"gold_kid={gold_kid_p:.6f}")
print(f"gold_adult={gold_adult_p:.6f}")

In [ ]:
train_dataset = Dataset.from_list(
    [
        {
            "prompt": [{"role": "user", "content": row["prompt"]}],
            "completion": [{"role": "assistant", "content": row["kid"]}],
        }
        for row in train_rows
    ]
)

print(train_dataset)

## Training

In [ ]:
for variable_name in ("trainer", "model"):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    seed=SEED,
    data_seed=SEED,
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,
    max_length=512,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

trainable_dtypes = {
    parameter.dtype
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
}

assert trainable_dtypes == {torch.float32}

train_result = trainer.train()

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print(train_result.metrics)
print(ADAPTER_DIR)

## Generation

In [ ]:
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

trainer.optimizer = None
trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "left"


@torch.inference_mode()
def generate_greedy(
    prompts: list[str],
    batch_size: int = 1,
    max_new_tokens: int = 160,
) -> list[str]:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

    answers = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        rendered = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for prompt in batch_prompts
        ]

        inputs = tokenizer(
            rendered,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=384,
        ).to("cuda")

        input_width = inputs["input_ids"].shape[1]

        generated = model.generate(
            **inputs,
            do_sample=False,
            num_beams=1,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        answers.extend(
            answer.strip()
            for answer in tokenizer.batch_decode(
                generated[:, input_width:],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
        )

    return answers


test_prompts = [row["prompt"] for row in style_test_rows]
generated_answers = generate_greedy(test_prompts)

assert len(generated_answers) == 50
assert all(generated_answers)

print(generated_answers[0])

## Metric

In [ ]:
p_simple_each = style_probabilities(generated_answers)
p_simple_mean = float(p_simple_each.mean())


def task1_interval(value: float) -> tuple[str, str]:
    if value < 0.4:
        return "А", "< 0.4"
    if value < 0.7:
        return "Г", "0.4 – 0.7"
    if value < 0.9:
        return "Б", "0.7 – 0.9"
    return "В", "0.9 – 1.0"


answer_letter, answer_interval = task1_interval(p_simple_mean)

results = pd.DataFrame(
    {
        "prompt": test_prompts,
        "generated_answer": generated_answers,
        "P_simple": p_simple_each,
    }
)

results.to_csv(
    OUTPUT_DIR / "public_test_style_predictions.csv",
    index=False,
    encoding="utf-8",
)

print(f"TASK_1_METRIC_P_SIMPLE={p_simple_mean:.6f}")
print(f"TASK_1_INTERVAL={answer_interval}")
print(f"TASK_1_ANSWER={answer_letter}")

# Задача 2

## DPO-датасет

In [ ]:
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

assert ADAPTER_DIR.exists()
assert "p_simple_mean" in globals()

task1_p_simple = float(p_simple_mean)

dpo_dataset = Dataset.from_list(
    [
        {
            "prompt": [
                {
                    "role": "user",
                    "content": row["prompt"],
                }
            ],
            "chosen": [
                {
                    "role": "assistant",
                    "content": row["kid"],
                }
            ],
            "rejected": [
                {
                    "role": "assistant",
                    "content": row["adult"],
                }
            ],
        }
        for row in train_rows
    ]
)

assert len(dpo_dataset) == 1489

print(dpo_dataset)
print(dpo_dataset[0])

## загрузка результата SFT

In [ ]:
for variable_name in (
    "trainer",
    "model",
    "base_model",
    "dpo_model",
    "dpo_trainer",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

dpo_model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=True,
)

dpo_model.config.use_cache = False
dpo_model.train()

trainable_params, all_params = (
    dpo_model.get_nb_trainable_parameters()
)

print(f"trainable={trainable_params}")
print(f"all={all_params}")

## DPO-обучение

In [ ]:
DPO_OUTPUT_DIR = (
    OUTPUT_DIR.parent / "task2_dpo"
)

DPO_ADAPTER_DIR = (
    DPO_OUTPUT_DIR / "adapter"
)

DPO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dpo_args = DPOConfig(
    output_dir=str(
        DPO_OUTPUT_DIR / "checkpoints"
    ),

    seed=SEED,
    data_seed=SEED,

    num_train_epochs=1,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,

    max_length=512,
    truncation_mode="keep_start",

    beta=0.1,
    loss_type="sigmoid",

    precompute_ref_log_probs=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

for parameter in (
    dpo_trainer.model.parameters()
):
    if parameter.requires_grad:
        parameter.data = (
            parameter.data.float()
        )

trainable_dtypes = {
    parameter.dtype
    for parameter
    in dpo_trainer.model.parameters()
    if parameter.requires_grad
}

assert trainable_dtypes == {
    torch.float32
}

assert (
    "ref"
    in dpo_trainer.model.peft_config
)

print(
    dpo_trainer.model.peft_config.keys()
)
print(trainable_dtypes)

dpo_result = dpo_trainer.train()

dpo_trainer.save_model(
    str(DPO_ADAPTER_DIR)
)

tokenizer.save_pretrained(
    str(DPO_ADAPTER_DIR)
)

print(dpo_result.metrics)
print(DPO_ADAPTER_DIR)

## генерация

In [ ]:
model = dpo_trainer.model

model.set_adapter("default")
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

dpo_trainer.optimizer = None
dpo_trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "left"

generated_answers_dpo = (
    generate_greedy(test_prompts)
)

assert len(
    generated_answers_dpo
) == 50

assert all(generated_answers_dpo)

print(generated_answers_dpo[0])

## итоговая метрика

In [ ]:
p_simple_dpo_each = (
    style_probabilities(
        generated_answers_dpo
    )
)

p_simple_dpo = float(
    p_simple_dpo_each.mean()
)


def task2_interval(
    value: float,
) -> tuple[str, str]:
    if value < 0.4:
        return "Г", "< 0.4"

    if value < 0.7:
        return "А", "0.4 – 0.7"

    if value < 0.9:
        return "В", "0.7 – 0.9"

    return "Б", "0.9 – 1.0"


task2_letter, task2_range = (
    task2_interval(p_simple_dpo)
)

results_dpo = pd.DataFrame(
    {
        "prompt": test_prompts,
        "generated_answer":
            generated_answers_dpo,
        "P_simple":
            p_simple_dpo_each,
    }
)

results_dpo.to_csv(
    DPO_OUTPUT_DIR
    / "public_test_style_predictions.csv",
    index=False,
    encoding="utf-8",
)

print(
    f"TASK_1_P_SIMPLE="
    f"{task1_p_simple:.6f}"
)

print(
    f"TASK_2_P_SIMPLE="
    f"{p_simple_dpo:.6f}"
)

print(
    f"TASK_2_DELTA="
    f"{p_simple_dpo - task1_p_simple:+.6f}"
)

print(
    f"TASK_2_INTERVAL="
    f"{task2_range}"
)

print(
    f"TASK_2_ANSWER="
    f"{task2_letter}"
)

# Задача 3

## данные

In [ ]:
GOOD_BAD_PATH = find_file(
    "good_bad.jsonl",
    "data",
)

QUALITY_TEST_PATH = find_file(
    "public_test_quality.jsonl",
    "data",
)

good_bad_rows = read_jsonl(
    GOOD_BAD_PATH
)

quality_test_rows = read_jsonl(
    QUALITY_TEST_PATH
)

assert len(good_bad_rows) == 2226
assert len(quality_test_rows) == 50

rm_train_dataset = Dataset.from_list(
    [
        {
            "prompt": [
                {
                    "role": "user",
                    "content": row["instruction"],
                }
            ],
            "chosen": [
                {
                    "role": "assistant",
                    "content": row["chosen"],
                }
            ],
            "rejected": [
                {
                    "role": "assistant",
                    "content": row["rejected"],
                }
            ],
        }
        for row in good_bad_rows
    ]
)

rm_test_dataset = Dataset.from_list(
    [
        {
            "prompt": [
                {
                    "role": "user",
                    "content": row["prompt"],
                }
            ],
            "chosen": [
                {
                    "role": "assistant",
                    "content": row["chosen"],
                }
            ],
            "rejected": [
                {
                    "role": "assistant",
                    "content": row["rejected"],
                }
            ],
        }
        for row in quality_test_rows
    ]
)

print(rm_train_dataset)
print(rm_test_dataset)
print(rm_train_dataset[0])

## освобождение памяти

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
)

from trl import (
    RewardConfig,
    RewardTrainer,
)

for variable_name in (
    "dpo_trainer",
    "dpo_model",
    "base_model",
    "model",
    "trainer",
    "reward_trainer",
    "reward_model",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

print(
    f"Allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"Reserved: "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

## reward-модель

In [ ]:
RM_OUTPUT_DIR = (
    OUTPUT_DIR.parent
    / "task3_reward_model"
)

RM_ADAPTER_DIR = (
    RM_OUTPUT_DIR
    / "adapter"
)

RM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

reward_tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
    )
)

reward_tokenizer.pad_token = (
    reward_tokenizer.eos_token
)

reward_tokenizer.padding_side = "right"

rm_quantization_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.float16
        ),
    )
)

reward_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_ID,
        num_labels=1,
        quantization_config=(
            rm_quantization_config
        ),
        dtype=torch.float16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )
)

reward_model.config.pad_token_id = (
    reward_tokenizer.pad_token_id
)

reward_model.config.use_cache = False
reward_model.config.problem_type = (
    "regression"
)

reward_model = (
    prepare_model_for_kbit_training(
        reward_model,
        use_gradient_checkpointing=True,
    )
)

rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=[
        "score",
    ],
)

print(type(reward_model))

## обучение Reward Model

In [ ]:
rm_args = RewardConfig(
    output_dir=str(
        RM_OUTPUT_DIR
        / "checkpoints"
    ),

    seed=SEED,
    data_seed=SEED,

    num_train_epochs=1,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    weight_decay=0.01,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,

    max_length=512,

    center_rewards_coefficient=1e-2,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    eval_strategy="no",
    save_strategy="no",

    logging_steps=10,
    report_to="none",

    dataloader_num_workers=0,
)

reward_trainer = RewardTrainer(
    model=reward_model,
    args=rm_args,
    train_dataset=rm_train_dataset,
    processing_class=reward_tokenizer,
    peft_config=rm_lora_config,
)

for parameter in (
    reward_trainer.model.parameters()
):
    if parameter.requires_grad:
        parameter.data = (
            parameter.data.float()
        )

trainable_dtypes = {
    parameter.dtype
    for parameter
    in reward_trainer.model.parameters()
    if parameter.requires_grad
}

assert trainable_dtypes == {
    torch.float32
}

trainable_params, all_params = (
    reward_trainer.model
    .get_nb_trainable_parameters()
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"All parameters: "
    f"{all_params:,}"
)

print(trainable_dtypes)

rm_train_result = (
    reward_trainer.train()
)

reward_trainer.save_model(
    str(RM_ADAPTER_DIR)
)

reward_tokenizer.save_pretrained(
    str(RM_ADAPTER_DIR)
)

print(rm_train_result.metrics)
print(RM_ADAPTER_DIR)

## подсчёт pairwise accuracy

In [ ]:
reward_model = reward_trainer.model

reward_model.gradient_checkpointing_disable()
reward_model.config.use_cache = True
reward_model.eval()

reward_trainer.optimizer = None
reward_trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

reward_tokenizer.padding_side = "right"


@torch.inference_mode()
def get_reward_scores(
    prompts: list[str],
    answers: list[str],
) -> np.ndarray:
    scores = []

    for prompt, answer in zip(
        prompts,
        answers,
    ):
        messages = [
            {
                "role": "user",
                "content": prompt,
            },
            {
                "role": "assistant",
                "content": answer,
            },
        ]

        text = (
            reward_tokenizer
            .apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        )

        inputs = reward_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to("cuda")

        output = reward_model(
            **inputs
        )

        score = float(
            output.logits
            .squeeze()
            .float()
            .item()
        )

        scores.append(score)

    return np.array(
        scores,
        dtype=np.float32,
    )


quality_prompts = [
    row["prompt"]
    for row in quality_test_rows
]

quality_chosen = [
    row["chosen"]
    for row in quality_test_rows
]

quality_rejected = [
    row["rejected"]
    for row in quality_test_rows
]

chosen_rewards = get_reward_scores(
    quality_prompts,
    quality_chosen,
)

rejected_rewards = get_reward_scores(
    quality_prompts,
    quality_rejected,
)

correct_pairs = (
    chosen_rewards
    > rejected_rewards
)

task3_accuracy = float(
    correct_pairs.mean()
)

task3_margin = float(
    (
        chosen_rewards
        - rejected_rewards
    ).mean()
)

print(
    f"Correct pairs: "
    f"{correct_pairs.sum()}/"
    f"{len(correct_pairs)}"
)

print(
    f"Mean margin: "
    f"{task3_margin:.6f}"
)

print(
    f"TASK_3_ACCURACY="
    f"{task3_accuracy:.6f}"
)

In [ ]:
def task3_interval(
    value: float,
) -> tuple[str, str]:
    if value < 0.6:
        return "Б", "< 0.6"

    if value < 0.75:
        return "В", "0.6 – 0.75"

    if value < 0.9:
        return "Г", "0.75 – 0.9"

    return "А", "0.9 – 1.0"


task3_letter, task3_range = (
    task3_interval(
        task3_accuracy
    )
)

results_rm = pd.DataFrame(
    {
        "prompt": quality_prompts,
        "chosen_reward": chosen_rewards,
        "rejected_reward": rejected_rewards,
        "correct": correct_pairs,
    }
)

results_rm.to_csv(
    RM_OUTPUT_DIR
    / "public_test_quality_scores.csv",
    index=False,
    encoding="utf-8",
)

print(
    f"TASK_3_ACCURACY="
    f"{task3_accuracy:.6f}"
)

print(
    f"TASK_3_INTERVAL="
    f"{task3_range}"
)

print(
    f"TASK_3_ANSWER="
    f"{task3_letter}"
)

# Задача 4

## Датасет для DPO по качеству

In [ ]:
quality_dpo_dataset = Dataset.from_list(
    [
        {
            "prompt": [
                {
                    "role": "user",
                    "content": row["instruction"],
                }
            ],
            "chosen": [
                {
                    "role": "assistant",
                    "content": row["chosen"],
                }
            ],
            "rejected": [
                {
                    "role": "assistant",
                    "content": row["rejected"],
                }
            ],
        }
        for row in good_bad_rows
    ]
)

assert len(quality_dpo_dataset) == 2226

print(quality_dpo_dataset)
print(quality_dpo_dataset[0])

## Освобождение памяти и загрузка модели задачи 2

In [ ]:
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

for variable_name in (
    "reward_trainer",
    "reward_model",
    "base_model",
    "dpo_model",
    "dpo_trainer",
    "model",
    "trainer",
    "quality_base_model",
    "quality_dpo_model",
    "task4_trainer",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

quality_quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

quality_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quality_quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

quality_base_model.config.use_cache = False

quality_base_model = prepare_model_for_kbit_training(
    quality_base_model,
    use_gradient_checkpointing=True,
)

quality_dpo_model = PeftModel.from_pretrained(
    quality_base_model,
    str(DPO_ADAPTER_DIR),
    is_trainable=True,
)

quality_dpo_model.config.use_cache = False
quality_dpo_model.train()

trainable_params, all_params = (
    quality_dpo_model.get_nb_trainable_parameters()
)

print(f"trainable={trainable_params:,}")
print(f"all={all_params:,}")

## DPO-обучение

In [ ]:
TASK4_OUTPUT_DIR = (
    OUTPUT_DIR.parent
    / "task4_quality_dpo"
)

TASK4_ADAPTER_DIR = (
    TASK4_OUTPUT_DIR
    / "adapter"
)

TASK4_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

task4_args = DPOConfig(
    output_dir=str(
        TASK4_OUTPUT_DIR
        / "checkpoints"
    ),

    seed=SEED,
    data_seed=SEED,

    num_train_epochs=1,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,

    max_length=512,
    truncation_mode="keep_start",

    beta=0.1,
    loss_type="sigmoid",

    precompute_ref_log_probs=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",

    remove_unused_columns=False,
    dataloader_num_workers=0,
)

task4_trainer = DPOTrainer(
    model=quality_dpo_model,
    ref_model=None,
    args=task4_args,
    train_dataset=quality_dpo_dataset,
    processing_class=tokenizer,
)

for parameter in task4_trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

trainable_dtypes = {
    parameter.dtype
    for parameter in task4_trainer.model.parameters()
    if parameter.requires_grad
}

assert trainable_dtypes == {torch.float32}
assert "ref" in task4_trainer.model.peft_config

print(task4_trainer.model.peft_config.keys())
print(trainable_dtypes)

task4_train_result = task4_trainer.train()

task4_trainer.model.save_pretrained(
    str(TASK4_ADAPTER_DIR),
    selected_adapters=["default"],
)

tokenizer.save_pretrained(
    str(TASK4_ADAPTER_DIR)
)

print(task4_train_result.metrics)
print(TASK4_ADAPTER_DIR)

## Функция средней logprob ответа

In [ ]:
import torch.nn.functional as F

task4_model = task4_trainer.model

task4_model.set_adapter("default")
task4_model.gradient_checkpointing_disable()
task4_model.config.use_cache = False
task4_model.eval()

task4_trainer.optimizer = None
task4_trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "right"


def encode_answer_span(
    prompt: str,
    answer: str,
) -> tuple[list[int], int, int]:
    prompt_messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    empty_messages = [
        {
            "role": "user",
            "content": prompt,
        },
        {
            "role": "assistant",
            "content": "",
        },
    ]

    full_messages = [
        {
            "role": "user",
            "content": prompt,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=False,
    )

    empty_ids = tokenizer.apply_chat_template(
        empty_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
    )

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
    )

    assert full_ids[:len(prompt_ids)] == prompt_ids
    assert empty_ids[:len(prompt_ids)] == prompt_ids

    suffix_ids = empty_ids[len(prompt_ids):]

    if suffix_ids:
        assert full_ids[-len(suffix_ids):] == suffix_ids
        answer_end = len(full_ids) - len(suffix_ids)
    else:
        answer_end = len(full_ids)

    answer_start = len(prompt_ids)

    assert answer_end > answer_start

    return full_ids, answer_start, answer_end

## Подсчёт length-normalized logprob

In [ ]:
@torch.inference_mode()
def pair_mean_logprobs(
    prompt: str,
    chosen: str,
    rejected: str,
) -> tuple[float, float]:
    chosen_ids, chosen_start, chosen_end = (
        encode_answer_span(
            prompt,
            chosen,
        )
    )

    rejected_ids, rejected_start, rejected_end = (
        encode_answer_span(
            prompt,
            rejected,
        )
    )

    encoded = tokenizer.pad(
        {
            "input_ids": [
                chosen_ids,
                rejected_ids,
            ]
        },
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = task4_model(
        **encoded,
        use_cache=False,
    )

    input_ids = encoded["input_ids"]
    shift_logits = outputs.logits[:, :-1, :].float()
    shift_labels = input_ids[:, 1:]

    token_logprobs = -F.cross_entropy(
        shift_logits.reshape(
            -1,
            shift_logits.shape[-1],
        ),
        shift_labels.reshape(-1),
        reduction="none",
    ).reshape(
        shift_labels.shape
    )

    positions = torch.arange(
        1,
        input_ids.shape[1],
        device=input_ids.device,
    )

    starts = torch.tensor(
        [
            chosen_start,
            rejected_start,
        ],
        device=input_ids.device,
    )

    ends = torch.tensor(
        [
            chosen_end,
            rejected_end,
        ],
        device=input_ids.device,
    )

    answer_mask = (
        (positions.unsqueeze(0) >= starts.unsqueeze(1))
        &
        (positions.unsqueeze(0) < ends.unsqueeze(1))
    )

    token_counts = answer_mask.sum(dim=1)

    assert bool(
        torch.all(token_counts > 0)
    )

    mean_logprobs = (
        (
            token_logprobs
            * answer_mask
        ).sum(dim=1)
        / token_counts
    )

    return (
        float(mean_logprobs[0].item()),
        float(mean_logprobs[1].item()),
    )

## Вычисление accuracy

In [ ]:
task4_chosen_logprobs = []
task4_rejected_logprobs = []

for index, row in enumerate(
    quality_test_rows,
    start=1,
):
    chosen_logprob, rejected_logprob = (
        pair_mean_logprobs(
            row["prompt"],
            row["chosen"],
            row["rejected"],
        )
    )

    task4_chosen_logprobs.append(
        chosen_logprob
    )

    task4_rejected_logprobs.append(
        rejected_logprob
    )

    if index % 10 == 0:
        print(f"{index}/50")

task4_chosen_logprobs = np.array(
    task4_chosen_logprobs,
    dtype=np.float32,
)

task4_rejected_logprobs = np.array(
    task4_rejected_logprobs,
    dtype=np.float32,
)

task4_correct = (
    task4_chosen_logprobs
    >
    task4_rejected_logprobs
)

task4_accuracy = float(
    task4_correct.mean()
)

task4_mean_margin = float(
    (
        task4_chosen_logprobs
        -
        task4_rejected_logprobs
    ).mean()
)

print(
    f"Correct pairs: "
    f"{task4_correct.sum()}/"
    f"{len(task4_correct)}"
)

print(
    f"Mean normalized logprob margin: "
    f"{task4_mean_margin:.6f}"
)

print(
    f"TASK_4_ACCURACY="
    f"{task4_accuracy:.6f}"
)

In [ ]:
def task4_interval(
    value: float,
) -> tuple[str, str]:
    if value < 0.6:
        return "Б", "< 0.6"

    if value < 0.75:
        return "Г", "0.6 – 0.75"

    if value < 0.9:
        return "В", "0.75 – 0.9"

    return "А", "0.9 – 1.0"


task4_letter, task4_range = (
    task4_interval(
        task4_accuracy
    )
)

task4_results = pd.DataFrame(
    {
        "prompt": [
            row["prompt"]
            for row in quality_test_rows
        ],
        "chosen_mean_logprob":
            task4_chosen_logprobs,
        "rejected_mean_logprob":
            task4_rejected_logprobs,
        "margin":
            task4_chosen_logprobs
            - task4_rejected_logprobs,
        "correct":
            task4_correct,
    }
)

task4_results.to_csv(
    TASK4_OUTPUT_DIR
    / "public_test_quality_logprobs.csv",
    index=False,
    encoding="utf-8",
)

print(
    f"TASK_4_ACCURACY="
    f"{task4_accuracy:.6f}"
)

print(
    f"TASK_4_INTERVAL="
    f"{task4_range}"
)

print(
    f"TASK_4_ANSWER="
    f"{task4_letter}"
)

# Задача 5

## Датасет

In [ ]:
simpo_dataset = Dataset.from_list(
    [
        {
            "prompt": [
                {
                    "role": "user",
                    "content": row["instruction"],
                }
            ],
            "chosen": [
                {
                    "role": "assistant",
                    "content": row["chosen"],
                }
            ],
            "rejected": [
                {
                    "role": "assistant",
                    "content": row["rejected"],
                }
            ],
        }
        for row in good_bad_rows
    ]
)

assert len(simpo_dataset) == 2226

print(simpo_dataset)
print(simpo_dataset[0])

## Очистка памяти и загрузка модели задачи 2

In [ ]:
from peft import PeftModel
from trl.experimental.cpo import CPOConfig, CPOTrainer

for variable_name in (
    "task4_trainer",
    "task4_model",
    "quality_dpo_model",
    "quality_base_model",
    "reward_trainer",
    "reward_model",
    "simpo_trainer",
    "simpo_model",
    "simpo_base_model",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

simpo_quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

simpo_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=simpo_quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

simpo_base_model.config.use_cache = False

simpo_base_model = prepare_model_for_kbit_training(
    simpo_base_model,
    use_gradient_checkpointing=True,
)

simpo_model = PeftModel.from_pretrained(
    simpo_base_model,
    str(DPO_ADAPTER_DIR),
    is_trainable=True,
)

simpo_model.config.use_cache = False
simpo_model.train()

trainable_params, all_params = (
    simpo_model.get_nb_trainable_parameters()
)

print(f"trainable={trainable_params:,}")
print(f"all={all_params:,}")
print(simpo_model.peft_config.keys())

## Обучение SimPO

In [ ]:
SIMPO_OUTPUT_DIR = (
    OUTPUT_DIR.parent
    / "task5_simpo"
)

SIMPO_ADAPTER_DIR = (
    SIMPO_OUTPUT_DIR
    / "adapter"
)

SIMPO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

simpo_args = CPOConfig(
    output_dir=str(
        SIMPO_OUTPUT_DIR
        / "checkpoints"
    ),

    seed=SEED,
    data_seed=SEED,

    num_train_epochs=1,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,

    max_length=512,

    loss_type="simpo",
    beta=2.0,
    simpo_gamma=0.5,
    cpo_alpha=0.0,
    label_smoothing=0.0,
    disable_dropout=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    logging_steps=10,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataset_num_proc=1,
)

simpo_trainer = CPOTrainer(
    model=simpo_model,
    args=simpo_args,
    train_dataset=simpo_dataset,
    processing_class=tokenizer,
)

for parameter in simpo_trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

trainable_dtypes = {
    parameter.dtype
    for parameter in simpo_trainer.model.parameters()
    if parameter.requires_grad
}

adapter_names = set(
    simpo_trainer.model.peft_config.keys()
)

assert trainable_dtypes == {torch.float32}
assert adapter_names == {"default"}

print(adapter_names)
print(trainable_dtypes)

torch.cuda.reset_peak_memory_stats()

simpo_train_result = simpo_trainer.train()

simpo_peak_memory_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

simpo_trainer.save_model(
    str(SIMPO_ADAPTER_DIR)
)

tokenizer.save_pretrained(
    str(SIMPO_ADAPTER_DIR)
)

print(simpo_train_result.metrics)
print(f"Peak allocated VRAM: {simpo_peak_memory_gb:.2f} GB")
print(SIMPO_ADAPTER_DIR)

## Подготовка модели к оценке

In [ ]:
import torch.nn.functional as F

simpo_eval_model = simpo_trainer.model

simpo_eval_model.set_adapter("default")
simpo_eval_model.gradient_checkpointing_disable()
simpo_eval_model.config.use_cache = False
simpo_eval_model.eval()

simpo_trainer.optimizer = None
simpo_trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "right"

## Границы токенов ответа

In [ ]:
def simpo_encode_answer_span(
    prompt: str,
    answer: str,
) -> tuple[list[int], int, int]:
    prompt_messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    empty_messages = [
        {
            "role": "user",
            "content": prompt,
        },
        {
            "role": "assistant",
            "content": "",
        },
    ]

    full_messages = [
        {
            "role": "user",
            "content": prompt,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=False,
    )

    empty_ids = tokenizer.apply_chat_template(
        empty_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
    )

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False,
    )

    assert full_ids[:len(prompt_ids)] == prompt_ids
    assert empty_ids[:len(prompt_ids)] == prompt_ids

    suffix_ids = empty_ids[len(prompt_ids):]

    if suffix_ids:
        assert full_ids[-len(suffix_ids):] == suffix_ids
        answer_end = len(full_ids) - len(suffix_ids)
    else:
        answer_end = len(full_ids)

    answer_start = len(prompt_ids)

    assert answer_start >= 1
    assert answer_end > answer_start

    return full_ids, answer_start, answer_end

## Средняя logprob токенов ответа

In [ ]:
@torch.inference_mode()
def simpo_mean_answer_logprob(
    prompt: str,
    answer: str,
) -> float:
    input_ids_list, answer_start, answer_end = (
        simpo_encode_answer_span(
            prompt,
            answer,
        )
    )

    input_ids = torch.tensor(
        [input_ids_list],
        dtype=torch.long,
        device="cuda",
    )

    attention_mask = torch.ones_like(
        input_ids
    )

    outputs = simpo_eval_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )

    shift_logits = (
        outputs.logits[0, :-1, :]
        .float()
    )

    shift_labels = input_ids[0, 1:]

    token_logprobs = -F.cross_entropy(
        shift_logits,
        shift_labels,
        reduction="none",
    )

    answer_logprobs = token_logprobs[
        answer_start - 1:
        answer_end - 1
    ]

    assert answer_logprobs.numel() == (
        answer_end - answer_start
    )

    return float(
        answer_logprobs.mean().item()
    )   

## Подсчёт accuracy

In [ ]:
simpo_chosen_logprobs = []
simpo_rejected_logprobs = []

for index, row in enumerate(
    quality_test_rows,
    start=1,
):
    chosen_logprob = (
        simpo_mean_answer_logprob(
            row["prompt"],
            row["chosen"],
        )
    )

    rejected_logprob = (
        simpo_mean_answer_logprob(
            row["prompt"],
            row["rejected"],
        )
    )

    simpo_chosen_logprobs.append(
        chosen_logprob
    )

    simpo_rejected_logprobs.append(
        rejected_logprob
    )

    if index % 10 == 0:
        print(f"{index}/50")

simpo_chosen_logprobs = np.array(
    simpo_chosen_logprobs,
    dtype=np.float32,
)

simpo_rejected_logprobs = np.array(
    simpo_rejected_logprobs,
    dtype=np.float32,
)

simpo_correct = (
    simpo_chosen_logprobs
    >
    simpo_rejected_logprobs
)

task5_accuracy = float(
    simpo_correct.mean()
)

task5_mean_margin = float(
    (
        simpo_chosen_logprobs
        -
        simpo_rejected_logprobs
    ).mean()
)

print(
    f"Correct pairs: "
    f"{simpo_correct.sum()}/"
    f"{len(simpo_correct)}"
)

print(
    f"Mean normalized logprob margin: "
    f"{task5_mean_margin:.6f}"
)

print(
    f"TASK_5_ACCURACY="
    f"{task5_accuracy:.6f}"
)

In [ ]:
def task5_interval(
    value: float,
) -> tuple[str, str]:
    if value < 0.6:
        return "Г", "< 0.6"

    if value < 0.75:
        return "В", "0.6 – 0.75"

    if value < 0.9:
        return "Б", "0.75 – 0.9"

    return "А", "0.9 – 1.0"


task5_letter, task5_range = (
    task5_interval(
        task5_accuracy
    )
)

task5_delta_vs_dpo = (
    task5_accuracy
    - task4_accuracy
)

simpo_results = pd.DataFrame(
    {
        "prompt": [
            row["prompt"]
            for row in quality_test_rows
        ],
        "chosen_mean_logprob":
            simpo_chosen_logprobs,
        "rejected_mean_logprob":
            simpo_rejected_logprobs,
        "margin":
            simpo_chosen_logprobs
            - simpo_rejected_logprobs,
        "correct":
            simpo_correct,
    }
)

simpo_results.to_csv(
    SIMPO_OUTPUT_DIR
    / "public_test_quality_logprobs.csv",
    index=False,
    encoding="utf-8",
)

print(
    f"TASK_4_DPO_ACCURACY="
    f"{task4_accuracy:.6f}"
)

print(
    f"TASK_5_SIMPO_ACCURACY="
    f"{task5_accuracy:.6f}"
)

print(
    f"TASK_5_DELTA_VS_DPO="
    f"{task5_delta_vs_dpo:+.6f}"
)

print(
    f"TASK_5_INTERVAL="
    f"{task5_range}"
)

print(
    f"TASK_5_ANSWER="
    f"{task5_letter}"
)